# Próg dopasowania frazy do nazwy klasy

Czyta gotowy pomiar `threshold` z `results/measurements/` i drukuje progi pewności przyjęte dla słownika każdego komponentu wraz z zasięgiem, jaki sygnał po tym progu osiąga. Sam nic nie liczy i nie ładuje żadnego modelu - próg powstaje w `scripts/measure_threshold.py`.

**Wymaga:** próby, oceny ręcznej i pomiaru, kolejno:

```powershell
# 1. cztery proby do oceny; domyslnie do data/interim/threshold/
python scripts/make_threshold_samples.py

# 2. ocena reczna, okolo 350 par, jeden klawisz na pare
python tools/judge/judge_pairs.py --samples data/interim/threshold/threshold_sample_*.csv --output data/interim/threshold/threshold_judgments.csv

# 3. przeniesienie ocen do katalogu adnotacji (krok RECZNY, zaden kod tam nie pisze)
Copy-Item data\interim\threshold\threshold_judgments.csv `
          data\annotations\threshold_judgments.csv

# 4. progi i zasiegi, per slownik
python scripts/measure_threshold.py --split dev
```

`judge_pairs.py` wymaga obu ścieżek i nie ma wartości domyślnych: `--samples` przyjmuje wzorzec plików próby, `--output` plik odpowiedzi. Klawisze: `Y` tak, `N` nie, `S` pomiń, `P` cofnij, `Q` zapisz i wyjdź. Plik odpowiedzi jest przepisywany po każdym naciśnięciu, więc przerwanie nic nie kosztuje, a ponowne uruchomienie wznawia od pierwszej nieocenionej pary. Reguły oceny są w `tools/judge/README.md`.

Próg każdego słownika jest najniższą pewnością, przy której wśród dopasowań przyjętych co najmniej 90% jest poprawnych. Obok progu stoi zasięg: odsetek fraz i odsetek zapytań, dla których dopasowanie próg osiąga. Próg wysoki przy zasięgu bliskim zeru znaczy sygnał poprawny i bezużyteczny, więc obie liczby czyta się razem. Para pominięta do końca jest w pliku ocen nieobecna, więc nie wchodzi do precyzji; jej udział podany jest osobno.

**Zapisuje:** nic, tylko wypisuje.

In [ ]:
import importlib
import json
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation import tables
from src.measurement import text_bridge
from src.utils import notebook

for module in (tables, text_bridge, notebook):
    importlib.reload(module)      # the kernel keeps a once-imported module in memory

VOCABULARIES = list(text_bridge.VOCABULARY_KIND)
POPULATION = Path(text_bridge.SAMPLE_DIR) / text_bridge.POPULATION_FILE

# The newest saved measurement. Every block below reads it and nothing here
# computes: the numbers were produced by scripts/measure_threshold.py, which is
# also what recorded the machine they came from.
records = notebook.load_measurements("threshold")
measurement = records[-1] if records else None
data = (measurement or {}).get("data", {})
population = (json.loads(POPULATION.read_text(encoding="utf-8"))
              if POPULATION.exists() else {})


def missing(*keys):
    """Which of these the measurement does not carry, so a block says what it
    needs instead of failing on a run that has not happened yet."""
    if not data:
        return ["pomiar threshold (uruchom scripts/measure_threshold.py --split dev)"]
    return [key for key in keys if not data.get(key)]


if measurement is None:
    print("Brak pomiaru. Uruchom: python scripts/measure_threshold.py --split dev")
else:
    material = data["material"]
    print(f"pomiar z {measurement['when']}, karta: "
          f"{measurement['machine'].get('gpu', '-')}")
    print(f"enkoder {material['encoder']}, poziom precyzji {text_bridge.PRECISION}")

    # The four thresholds stand on the hand-judged samples and on nothing else,
    # so that is what the first thing the notebook prints has to describe. The
    # material of `data["material"]` belongs to the OLD path -- the clips whose
    # Kinetics class is one of the 400 -- and reporting it here read as if it
    # were the basis of the thresholds below it.
    judged = data.get("judged") or {}
    if not judged:
        print("Brak ocen recznych w tym pomiarze - progi sie nie policzyly. "
              "Kroki 1-3 z naglowka notatnika, potem measure_threshold.py.")
    else:
        print("proby ocenione recznie - na nich stoja progi:")
        for name in VOCABULARIES:
            entry = judged.get(name)
            if entry is None:
                print(f"  {name:<26}brak w tym pomiarze")
                continue
            # a stratum the population file does not size (a literal match in a
            # vocabulary drawn whole) carries None and is left out of the sum
            drawn = sum(s["population"] for s in (entry.get("strata") or {}).values()
                        if s.get("population") is not None)
            print(f"  {name:<26}{entry['n_judged']} par ocenionych"
                  f" z {entry['n_sample'] or '?'} wylosowanych"
                  + (f" z populacji {drawn} par" if drawn else ""))

## 1. Skład próby ocenianej ręcznie

Na czym stoi kalibracja: ile fraz każdy słownik dał na wejściu, do ilu różnych par one się zwijają i ile z tych par poszło do oceny. Warstwa `1,0` to dopasowania dosłowne, gdzie jedna z postaci frazy jest nazwą klasy znak w znak i wektory są identyczne; reszta zakresu jest dzielona na dziesięć równych przedziałów. Liczby biorą się z `threshold_population.json`, nie z pomiaru.

In [ ]:
if not population:
    print(f"Brak {POPULATION}. Uruchom: python scripts/make_threshold_samples.py")
else:
    rows = []
    for name in VOCABULARIES:
        entry = population.get(name)
        if entry is None:
            rows.append([f"`{name}`"] + ["-"] * 6)
            continue
        held, drawn = entry["strata"], entry["sampled"]
        edges = entry["edges"]
        # a population small enough to judge whole is not stratified at all, so
        # there is no layer to report - a dash, not a zero
        whole = "all" in held
        rows.append([
            f"`{name}`", entry["n_texts"], entry["n_occurrences"], entry["n_pairs"],
            "-" if whole else f"{drawn.get('exact', 0)} z {held['exact']}",
            drawn["all"] if whole
            else sum(n for band, n in drawn.items() if band != "exact"),
            f"{tables.number(edges[0], 3)}-{tables.number(edges[-1], 3)}"
            if edges else "cala populacja, bez warstw",
        ])

    tables.show("Sklad proby do oceny recznej",
                ["Slownik", "Tekstow", "Wystapien fraz", "Roznych par",
                 "Z warstwy 1,0", "Z reszty zakresu", "Zakres przedzialow"],
                rows,
                note="Ocenia sie rozne PARY (fraza, klasa), nie wystapienia: powtorzona "
                     "fraza zawsze dostaje te sama klase przy tej samej pewnosci. "
                     "Zasieg nizej liczy sie juz na wystapieniach. Kreska w kolumnie "
                     "warstwy: populacja byla dosc mala, zeby ocenic ja w calosci, "
                     "wiec nie ma czego warstwowac ani wazyc.")

    bins = [f"b{i:02d}" for i in range(1, text_bridge.SAMPLE_BINS + 1)]
    rows = [[f"`{name}`"] + [f"{population[name]['sampled'].get(b, 0)} z "
                             f"{population[name]['strata'].get(b, 0)}" for b in bins]
            for name in VOCABULARIES
            if name in population and "all" not in population[name]["strata"]]
    if rows:
        tables.show("Rozklad po dziesieciu przedzialach (wylosowane z populacji)",
                    ["Slownik"] + bins, rows,
                    note="Przedzialy sa rowne co do SZEROKOSCI, nie co do liczebnosci: "
                         "pusty przedzial jest faktem o slowniku, ktory ma byc widoczny. "
                         "Przedzial ubozszy niz jego przydzial oddaje reszte pozostalym.")

## 2. Próg dopasowania

Próg każdego słownika policzony z ocen ręcznych: najniższa pewność, przy której co najmniej 90% przyjętych par jest poprawnych. Precyzja jest ważona warstwami, bo próba nie jest proporcjonalna do populacji.

In [ ]:
absent = missing("judged")
if absent:
    print("Brak ocen recznych. Kolejno:")
    print("  python scripts/make_threshold_samples.py")
    print("  python tools/judge/judge_pairs.py --samples data/interim/threshold/threshold_sample_*.csv --output data/interim/threshold/threshold_judgments.csv")
    print("  Copy-Item data\\interim\\threshold\\threshold_judgments.csv "
          "data\\annotations\\threshold_judgments.csv")
    print("  python scripts/measure_threshold.py --split dev")
else:
    rows = []
    for name in VOCABULARIES:
        entry = data["judged"].get(name)
        if entry is None:
            rows.append([f"`{name}`"] + ["-"] * 7)
            continue
        rows.append([
            f"`{name}`", f"{entry['n_judged']} z {entry['n_sample'] or '?'}",
            # one cell, not two: "yes / no / unresolved". The parentheses say so,
            # because without them a linter reads the pair as a forgotten comma
            (f"{entry['n_yes']} / {entry['n_no']} / "
             f"{tables.number(entry['n_unresolved'], 0)}"),
            tables.number(entry["threshold"], 3),
            tables.percent(entry["precision_at_threshold"]),
            f"{entry['n_above']}" + ("" if entry["enough_above"] else " (malo)"),
            tables.percent(entry["coverage_phrases"]),
            tables.percent(entry["coverage_queries"]),
        ])

    tables.show("Prog dopasowania frazy do nazwy klasy",
                ["Slownik", "Ocenionych par", "tak / nie / nierozstrzygnietych",
                 "Prog", "Precyzja [%]", "Par >= progu", "Zasieg fraz [%]",
                 "Zasieg zapytan [%]"], rows,
                note=f"Precyzja liczona na parach przyjetych, wazona warstwami. Para "
                     f"pominieta nie ma wiersza w pliku ocen, wiec nierozstrzygniete "
                     f"licza sie wzgledem liczebnosci proby, nie wzgledem ocen, ktore "
                     f"wrocily. Ponizej {text_bridge.MIN_ABOVE} par powyzej progu "
                     f"przedzial wokol {text_bridge.PRECISION} jest szerszy niz "
                     f"+/- 0,10 i kryterium nic nie mowi.")

    for name in VOCABULARIES:
        entry = data["judged"].get(name) or {}
        if entry.get("reason"):
            print(f"{name}: {entry['reason']}")
        if entry.get("unresolved_high"):
            print(f"{name}: par nierozstrzygnietych "
                  f"{tables.number(entry['n_unresolved'], 0)} z "
                  f"{tables.number(entry['n_sample'], 0)}, czyli "
                  f"{tables.percent(entry['unresolved_share'])} % - wiecej niz jedna "
                  f"na dwadziescia, wiec to sygnal o pytaniu albo o materiale")

## 3. Aktywacja sygnałów na zbiorach deweloperskich

Odsetek zapytań, o których każdy sygnał w ogóle się wypowie: fraza właściwego rodzaju istnieje i jej dopasowanie osiąga próg. Dwa wiersze bez progu, `query_time_detection` i `face_regions`, pytają tylko o istnienie frazy, bo mechanizmy otwartego słownika odpowiadają na każdą. Kreska znaczy "nie pytano", nie "zero".

In [ ]:
absent = missing("activation_dev")
if absent:
    print(f"Brak tabeli: {', '.join(absent)}.")
else:
    activation = data["activation_dev"]
    columns = sorted(next(iter(activation.values())))
    rows = [[f"`{signal}`"] + [tables.percent(per_dataset[dataset])
                               for dataset in columns]
            for signal, per_dataset in activation.items()]

    tables.show("Przewidywana aktywacja na dev",
                ["Sygnal"] + [c.upper() for c in columns], rows,
                note="Kreska: pytania nie zadano - albo brak pliku zapytan zbioru, albo "
                     "slownik nie ma jeszcze progu. Zero znaczyloby, ze zapytano i nic "
                     "nie przeszlo.")

## 4. Frazy poniżej progu

Najczęstsze głowy fraz, których dopasowanie progu nie osiągnęło, czyli to, o czym każdy sygnał milczy.

In [ ]:
absent = missing("rejected_phrases")
if absent:
    print(f"Brak tabeli: {', '.join(absent)}.")
else:
    rows = [[f"`{name}`",
             ", ".join(f"{head} ({count})" for head, count in entries[:5]) or "-"]
            for name, entries in data["rejected_phrases"].items()]
    tables.show("Frazy ponizej progu",
                ["Slownik", "Piec najczestszych glow (liczba wystapien)"], rows,
                align="ll",
                note="Liczy sie wystapienia fraz w zapytaniach deweloperskich, "
                     "nie rozne frazy.")

## 5. Stara ścieżka pomiaru i jej sufit

Próg liczony wobec etykiety Kinetics klipu, czyli ścieżka sprzed oceny ręcznej. Zostaje w raporcie jako zaniżona, bo opis i etykieta często nazywają tę samą czynność innymi słowami, a klip bywa etykietowany inną czynnością niż ta, o której mówi opis. Skalę widać na dopasowaniach dosłownych: tam poprawność jest z definicji, więc każda niezgodność obciąża etykietę.

In [ ]:
absent = missing("top1", "z")
if absent:
    print(f"Brak tabeli: {', '.join(absent)}.")
else:
    literal = data.get("literal_agreement") or {}
    # the material of THIS path, said here and not in the header: the filter that
    # produces it (the clip's class among the 400) is what the old truth needs
    # and what the calibration must not have, so the two counts describe two
    # different collections and neither is a mistake in the other
    calibration = ((data.get("judged") or {}).get(text_bridge.KINETICS) or {}).get("n_texts")
    print(f"Material starej sciezki: {data['material']['descriptions']} opisow "
          f"klipow, ktorych klasa zrodlowa nalezy do Kinetics-400"
          + (f" (kalibracja progow stoi na {calibration} opisach, bez tego filtra)"
             if calibration else "") + ".")
    print(f"Zgodnosc z etykieta klipu przy dopasowaniu doslownym: "
          f"{tables.number(literal.get('agreement'), 3)} na {literal.get('n', 0)} "
          f"parach. To jest sufit precyzji osiagalnej ta droga.")

    rows = [[f"`{name}`", data[name]["n_descriptions"], data[name]["n_matched"],
             tables.number(data[name]["precision_without_threshold"], 3),
             tables.number(data[name]["tau"], 3),
             tables.number(data[name]["precision_at_tau"], 3),
             tables.percent(data[name]["coverage"])]
            for name in ("top1", "z")]
    tables.show("Stara sciezka: prog wobec etykiety klipu",
                ["Miara", "Opisow", "Z dopasowaniem", "Precyzja bez progu", "tau",
                 "Precyzja przy tau", "Zasieg [%]"], rows,
                note="Jednostka oceny jest OPIS, nie fraza: klip ma jedna etykiete, "
                     "wiec najwyzej jedna fraza opisu moze byc trafna.")

    choice = data["measure_choice"]
    print(f"\nWybrana miara: {data['chosen_measure']} - {choice['reason']}")
    for name in ("top1", "z"):
        if data[name]["tau"] is None:
            print(f"{name}: {data[name]['reason']}")

    diagnostic = data["diagnostics"]
    print(f"\nSufit precyzji przy ocenie NA FRAZE: "
          f"{tables.number(diagnostic['per_phrase_ceiling'], 3)} "
          f"({diagnostic['n_phrases']} fraz czynnosciowych).")
    print(f"Odsetek opisow, w ktorych prog przechodzi takze fraza inna niz najlepsza: "
          f"{tables.percent(diagnostic['second_phrase_passes'])} %.")